In [1]:
import boto3
import pandas as pd
import numpy as np
# Conecta no S3 na região correta
s3 = boto3.client("s3", region_name="us-east-1")

bucket_name = "luis-sprint4"
file_key = "dados_limpos/InternacoesHospitalares_limpo.csv"

# Baixa o arquivo
obj = s3.get_object(Bucket=bucket_name, Key=file_key)

# Lê no Pandas
df = pd.read_csv(obj["Body"])

# Converte a coluna de data
df["data_internacao"] = pd.to_datetime(df["data_internacao"])

print(df.head())


      data_internacao       especialidade         municipio  idade sexo
0 2023-01-01 01:12:00  MEDICINA INTENSIVA             NATAL     42    F
1 2023-01-01 05:09:00         CARDIOLOGIA           Macaíba     45    M
2 2023-01-01 10:26:00          NEUROLOGIA  JARDIM DO SERIDÓ     71    F
3 2023-01-02 07:19:00   ONCOLOGIA CLÍNICA             NATAL     63    M
4 2023-01-02 07:54:00   CIRURGIA VASCULAR             NATAL     66    F


In [2]:
# Pacientes do sexo feminino, com mais de 60 anos, em Natal
filtro = df[(df["sexo"] == "F") & (df["idade"] > 60) & (df["municipio"] == "NATAL")]
filtro.head()


,data_internacao,especialidade,municipio,idade,sexo
4,2023-01-02 07:54:00,CIRURGIA VASCULAR,NATAL,66,F
18,2023-01-02 16:17:00,CIRURGIA ONCOLOGICA,NATAL,72,F
23,2023-01-03 06:58:00,ONCOLOGIA HEMATOLOGIA,NATAL,66,F
33,2023-01-03 12:11:00,PNEUMOLOGIA,NATAL,69,F
40,2023-01-04 07:23:00,ONCOLOGIA CLÍNICA,NATAL,62,F


In [3]:
# Quantidade de internações por especialidade
agg_especialidade = df.groupby("especialidade").size().reset_index(name="qtd_internacoes")
agg_especialidade.head()


,especialidade,qtd_internacoes
0,CARDIOLOGIA,661
1,CARDIOLOGIA PEDIÁTRICA,8
2,CIRURGIA BARIATRICA,77
3,CIRURGIA CARDIOVASCULAR,53
4,CIRURGIA DE CABEÇA E PESCOÇO,192


In [4]:
# Criar coluna categorizando pacientes como 'Idoso' (>=60) ou 'Adulto'
df["faixa_etaria"] = np.where(df["idade"] >= 60, "Idoso", "Adulto")
df[["idade", "faixa_etaria"]].head()


,idade,faixa_etaria
0,42,Adulto
1,45,Adulto
2,71,Idoso
3,63,Idoso
4,66,Idoso


In [5]:
# Converter idade para string e depois voltar pra int
df["idade_str"] = df["idade"].astype(str)
df["idade_convertida"] = pd.to_numeric(df["idade_str"])
df[["idade", "idade_str", "idade_convertida"]].head()


,idade,idade_str,idade_convertida
0,42,42,42
1,45,45,45
2,71,71,71
3,63,63,63
4,66,66,66


In [6]:
# Criar coluna com o mês da internação
df["mes_internacao"] = df["data_internacao"].dt.month
df[["data_internacao", "mes_internacao"]].head()


,data_internacao,mes_internacao
0,2023-01-01 01:12:00,1
1,2023-01-01 05:09:00,1
2,2023-01-01 10:26:00,1
3,2023-01-02 07:19:00,1
4,2023-01-02 07:54:00,1


In [7]:
# Filtrar especialidades que contenham a palavra 'CLÍNICA'
clinicas = df[df["especialidade"].str.contains("CLÍNICA", case=False, na=False)]
clinicas.head()


,data_internacao,especialidade,municipio,idade,sexo,faixa_etaria,idade_str,idade_convertida,mes_internacao
3,2023-01-02 07:19:00,ONCOLOGIA CLÍNICA,NATAL,63,M,Idoso,63,63,1
12,2023-01-02 11:24:00,ONCOLOGIA CLÍNICA,LAJES,61,M,Idoso,61,61,1
27,2023-01-03 08:35:00,ONCOLOGIA CLÍNICA,NATAL,49,F,Adulto,49,49,1
40,2023-01-04 07:23:00,ONCOLOGIA CLÍNICA,NATAL,62,F,Idoso,62,62,1
41,2023-01-04 07:50:00,ONCOLOGIA CLÍNICA,SANTO ANTÔNIO,74,F,Idoso,74,74,1
